# 课后练习解答（04.05_yolo_nms_custom_operator_development）

本解答对应《YOLO NMS 自定义后处理算子开发》课后练习，共 15 题。


### 问题1（单选题）

**题目：** 本实验最终选择哪条路线实现可跑通的 NMS 自定义算子？

A. Ascend C + ACLNN
B. 只使用 PyPTO
C. 只使用 TileLang
D. 只使用 CPU NumPy

**解答：** A

**解析：** 本实验真实跑通的是 Ascend C 自定义算子，并通过 ACLNN 调用验证。


### 问题2（单选题）

**题目：** 本实验中 `YoloNmsCustom` 的两个主要输出是？

A. keep 和 count
B. onnx 和 om
C. loss 和 lr
D. rank 和 world_size

**解答：** A

**解析：** keep 保存保留框索引，count 保存有效数量。


### 问题3（单选题）

**题目：** `msopgen gen` 的作用更接近哪一项？

A. 根据算子描述生成自定义算子工程骨架
B. 运行 YOLO OM 推理
C. 采集 msprof 数据
D. 压缩数据集

**解答：** A

**解析：** msopgen 用于生成算子工程基础结构。


### 问题4（单选题）

**题目：** 开发 Ascend C 自定义算子时，`op_host` 通常负责什么？

A. 算子定义、shape 推导和 tiling 等 host 侧逻辑
B. 读取 bus.jpg
C. Git LFS 上传
D. Jupyter 渲染

**解答：** A

**解析：** host 侧代码负责算子注册、推导、tiling 等准备工作。


### 问题5（多选题）

**题目：** 本节提到的三种自定义算子路线包括哪些？

A. Ascend C
B. PyPTO
C. TileLang
D. Excel VBA

**解答：** A、B、C

**解析：** 三者都是面向算子开发的不同路线，实验最终选择 Ascend C。


### 问题6（多选题）

**题目：** 一个完整 Ascend C 自定义算子工程通常包含哪些关键部分？

A. 算子工程描述/JSON
B. op_host
C. op_kernel
D. CMakePresets/build 脚本

**解答：** A、B、C、D

**解析：** 这些文件共同完成定义、编译和打包。


### 问题7（多选题）

**题目：** 编译安装自定义 OPP 后，应检查哪些产物或位置？

A. .o kernel 二进制
B. .json 算子信息
C. custom_opp_ubuntu_aarch64.run
D. opp/vendors/customize 安装目录

**解答：** A、B、C、D

**解析：** 这些证据说明算子已编译、打包并安装到 CANN OPP 目录。


### 问题8（判断题）

**题目：** PyPTO 和 TileLang 在本实验中是必须全部实现的三种算子之一。

**解答：** 错误

**解析：** 它们是可选路线或扩展认知；本实验只需选择 Ascend C 路线跑通。


### 问题9（判断题）

**题目：** 最小验证版 NMS kernel 可以先只做单 AI Core 串行实现，再进一步优化并行性能。

**解答：** 正确

**解析：** 先验证功能正确，再逐步做性能优化，是更稳妥的算子开发策略。


### 问题10（填空题）

**题目：** 本实验自定义算子的 opType 名称是 `____`。

**解答：** YoloNmsCustom

**解析：** 后续 ACLNN 接口和 OPP 安装都围绕这个 opType。


### 问题11（填空题）

**题目：** 自定义 OPP run 包安装成功后，常见提示是 `____`。

**解答：** SUCCESS

**解析：** 安装日志中的 SUCCESS 是重要验收信号。


### 问题12（简答题）

**题目：** 为什么本实验不建议在旧 AddCustom 工程上反复改名，而是新建 YoloNmsCustom 工程？

**解答：** 旧工程中可能残留 AddCustom、add_custom_0、tiling 注册和 CMake target 名称，容易造成注册混乱。新建工程能让 opType 从一开始就是 YoloNmsCustom，减少隐性错误。

**解析：** 这是之前反复排错后得到的工程经验。


### 问题13（简答题）

**题目：** 为什么需要 tiling 数据？

**解答：** tiling 数据把 host 侧计算出的规模、阈值、block 信息等传给 device kernel，使 kernel 能根据输入规模正确划分和执行计算。

**解析：** Ascend C 算子通常通过 tiling 连接 host 策略和 device 执行。


### 问题14（简答题）

**题目：** 如何判断自定义算子不是只编译成功，而是真的可调用？

**解答：** 除了看到 .o/.json 和 run 包，还要安装 custom OPP，并通过 ACLNN 调用工程执行 aclnnYoloNmsCustomGetWorkspaceSize 和 aclnnYoloNmsCustom，最后比较 keep/count 与 CPU baseline 是否一致。

**解析：** 编译成功只是工程构建完成，调用验证才证明算子进入运行链路。


### 问题15（代码设计题）

**题目：** 写出 ACLNN 调用自定义算子的核心两步接口顺序。

**解答：**

```cpp
uint64_t workspaceSize = 0;
aclOpExecutor *executor = nullptr;
auto ret = aclnnYoloNmsCustomGetWorkspaceSize(
    boxesTensor, scoresTensor, 0.45, 300, keepTensor, countTensor,
    &workspaceSize, &executor);
// allocate workspace if needed
ret = aclnnYoloNmsCustom(workspace, workspaceSize, executor, stream);
```

**解析：** 先获取 workspace 和 executor，再调用真正执行接口，这是 ACLNN 自定义算子调用的核心结构。
